In [36]:
import pandas as pd
from pathlib import Path
from urllib.parse import urlparse

In [37]:
df_iscx = pd.read_csv(Path.cwd().parent /'iscx2016kaggle.csv')
df_iscx = df_iscx[['url', 'type']]
df_iscx = df_iscx.rename(columns={'type': 'label'})
print("Columns after initial cleanup:")
print(df_iscx.head())

label_mapping = {
    'benign': 0,
    'phishing': 1,
    'defacement': 1,
    'malware': 1
}

df_iscx['label'] = df_iscx['label'].map(label_mapping)

# Check the final counts to ensure it's balanced [cite: 226, 227]
print("\nFinal Unified Label Counts:")
print(df_iscx['label'].value_counts())

# Preview the clean data
print("\nCleaned ISCX Dataframe:")
print(df_iscx.head())


# Balancing Dataset not done yet !!!

df_iscx_malicious = df_iscx[df_iscx['label']==1].copy()
print(df_iscx_malicious.head())
print(f"Total ISCX Malicious URLs extracted: {len(df_iscx_malicious)}")


Columns after initial cleanup:
                                                 url       label
0                                   br-icloud.com.br    phishing
1                mp3raid.com/music/krizz_kaliko.html      benign
2                    bopsecrets.org/rexroth/cr/1.htm      benign
3  http://www.garage-pirenne.be/index.php?option=...  defacement
4  http://adventure-nicaragua.net/index.php?optio...  defacement

Final Unified Label Counts:
label
0    428103
1    223088
Name: count, dtype: int64

Cleaned ISCX Dataframe:
                                                 url  label
0                                   br-icloud.com.br      1
1                mp3raid.com/music/krizz_kaliko.html      0
2                    bopsecrets.org/rexroth/cr/1.htm      0
3  http://www.garage-pirenne.be/index.php?option=...      1
4  http://adventure-nicaragua.net/index.php?optio...      1
                                                  url  label
0                                    br-icloud.c

In [38]:
df_tranco = pd.read_csv(Path.cwd().parent /'tranco_6G89X.csv', header=None)

df_tranco = df_tranco[[1]]

# Matching our unified format
df_tranco.columns = ['url']

# These are safe sites so label is 0
df_tranco['label'] = 0

print(df_tranco.head())

print(f"Total safe URLs added: {len(df_tranco)}")
#Balancing dataset not done yet !!

df_tranco_sampled = df_tranco.sample(n=300000, random_state=42)

print(f"Tranco sampled rows: {len(df_tranco_sampled)}")

                url  label
0        google.com      0
1  gtld-servers.net      0
2       gstatic.com      0
3    cloudflare.com      0
4      facebook.com      0
Total safe URLs added: 1000000
Tranco sampled rows: 300000


In [39]:
df_urlhaus = pd.read_csv(Path.cwd().parent /'urlhaus.csv',skiprows=8)


df_urlhaus = df_urlhaus[['url']]
df_urlhaus['label'] = 1

df_urlhaus['url'] = df_urlhaus['url'].str.strip()
print(df_urlhaus.head())

print(f"Total URLhaus malicious URLs: {len(df_urlhaus)}")

                                                 url  label
0                      http://112.93.202.244:48456/i      1
1                       http://221.1.227.205:35291/i      1
2  https://binary-block-tabel-expert-get.wiki/92c...      1
3                 http://112.93.202.244:48456/bin.sh      1
4  https://binary-block-state-collection.wiki/f96...      1
Total URLhaus malicious URLs: 11965


In [40]:
df_phishtank = pd.read_csv(Path.cwd().parent /'phishtank.csv')
df_phishtank = df_phishtank[['url']]
df_phishtank['label'] = 1
df_phishtank['url'] = df_phishtank['url'].str.strip()

print(df_phishtank.head())
print(f"Total Phistank malicious URLs: {len(df_phishtank)}")

                                                 url  label
0  https://gorillatourrwanda.com/htaccess/?mxid=b...      1
1  https://benmed1778652514257.2301823.meusitehos...      1
2                https://yavlixo.cfd/mobi/index.html      1
3                   https://appsbizzhost.com.au/capo      1
4                          https://shorturl.at/jJgJg      1
Total Phistank malicious URLs: 60110


In [41]:
df_iscx_benign = df_iscx[df_iscx['label'] == 0].sample(n=100000, random_state=42).copy()
master_df = pd.concat([df_tranco_sampled, df_iscx_benign, df_urlhaus, df_phishtank, df_iscx_malicious])
print(master_df.head())

print("Dataset Balance:")
print(master_df['label'].value_counts())


master_df.drop_duplicates(subset='url', keep='first', inplace=True)

print("Dataset Balance:")
print(master_df['label'].value_counts())
def normalize_url(url):
    try:
        url = str(url).strip()
        if not url.startswith('http'):
            url = 'http://' + url
        parsed = urlparse(url)
        result = parsed.netloc + parsed.path
        return result.rstrip('/')
    except Exception:
        return None 


master_df['url'] = master_df['url'].apply(normalize_url)

# Drop the bad IPv6 URLs
before = len(master_df)
master_df = master_df.dropna(subset=['url'])
master_df = master_df[master_df['url'].str.len() > 0]
after = len(master_df)

print(master_df.head())
master_df.to_csv(Path.cwd().parent /'master_dataset_clean.csv', index=False)

                            url  label
987231              noxlogic.nl      0
79954            dubaiexch.live      0
567130  brightroulettegroup.com      0
500891   worldglobalmedia.co.uk      0
55399           fobo-friends.ru      0
Dataset Balance:
label
0    400000
1    295163
Name: count, dtype: int64
Dataset Balance:
label
0    399999
1    285050
Name: count, dtype: int64
                            url  label
987231              noxlogic.nl      0
79954            dubaiexch.live      0
567130  brightroulettegroup.com      0
500891   worldglobalmedia.co.uk      0
55399           fobo-friends.ru      0


In [42]:
# CHECKING 

print("Sample URLs after normalization:")
print(master_df['url'].head(20).tolist())
print("\nLabel distribution:")
print(master_df['label'].value_counts())

# Check how many still don't have http
no_scheme = master_df[~master_df['url'].str.startswith('http')]
print(f"\nURLs still without scheme: {len(no_scheme)}")
print(no_scheme['url'].head(10).tolist())

Sample URLs after normalization:
['noxlogic.nl', 'dubaiexch.live', 'brightroulettegroup.com', 'worldglobalmedia.co.uk', 'fobo-friends.ru', 'anpdm.com', 'gdk.com', 'nasaacin.org', 'despegar.com.mx', 'cinemasrgfm.com', 'clearjournal.ca', 'multi-point.net', 'salatcalendar.com', 'rolorealm.com', 'anonaddy.com', 'okiu.ac.jp', 'quickdns.dk', 'tentsile.com', 'nayomi.com', 'ssp.se.gov.br']

Label distribution:
label
0    399999
1    285033
Name: count, dtype: int64

URLs still without scheme: 684965
['noxlogic.nl', 'dubaiexch.live', 'brightroulettegroup.com', 'worldglobalmedia.co.uk', 'fobo-friends.ru', 'anpdm.com', 'gdk.com', 'nasaacin.org', 'despegar.com.mx', 'cinemasrgfm.com']
